In [16]:
import torch
import torch.nn as nn

# Demo dimensions
T = 3
B = 5
T_cached = 7
T_pad = torch.tensor([0, 0, 1, 3, 8])

# Objective
Our goal is to creaet a (T, T+T_cached) (then broadcasted) that has these
- Causal: anything above the diagonal is 0 (t will not attend to anything beyond t)
- Padding: anything in the first T_pad[b] columns is 0, except the diagonal is 1 
    - Diagonals are still 1 so we don't screw up the softmax
    - It doesn't affect anything else since no real tokens later attend to them

In [17]:
tri = torch.ones([B, 1, T, T]).tril() # lower triangular region
fill = torch.ones([B, 1, T, T_cached]) # rectangular region
mask = torch.cat([fill, tri], dim=-1)

t_idx = torch.arange(T_cached+T).unsqueeze(0).repeat(T, 1)[None, None, :]
t_idx

tensor([[[[0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
          [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
          [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]]]])

In [18]:
padding_ends = T_pad[:, None, None, None]  # (B, 1, 1, 1)
(t_idx < padding_ends).shape  # Boolean tensor

torch.Size([5, 1, 3, 10])

In [19]:
mask.shape

torch.Size([5, 1, 3, 10])

In [20]:
mask[t_idx < padding_ends] = 0  # set first T_pad[b] columns to 0
mask.diagonal(offset=T_cached, dim1=-2, dim2=-1).fill_(1)  # set diagonals back to 1
mask

tensor([[[[1., 1., 1., 1., 1., 1., 1., 1., 0., 0.],
          [1., 1., 1., 1., 1., 1., 1., 1., 1., 0.],
          [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]]],


        [[[1., 1., 1., 1., 1., 1., 1., 1., 0., 0.],
          [1., 1., 1., 1., 1., 1., 1., 1., 1., 0.],
          [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]]],


        [[[0., 1., 1., 1., 1., 1., 1., 1., 0., 0.],
          [0., 1., 1., 1., 1., 1., 1., 1., 1., 0.],
          [0., 1., 1., 1., 1., 1., 1., 1., 1., 1.]]],


        [[[0., 0., 0., 1., 1., 1., 1., 1., 0., 0.],
          [0., 0., 0., 1., 1., 1., 1., 1., 1., 0.],
          [0., 0., 0., 1., 1., 1., 1., 1., 1., 1.]]],


        [[[0., 0., 0., 0., 0., 0., 0., 1., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 1., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 1., 1.]]]])